# Projeto:
**Detector de Fraudes - uma comparação entre o algorítmo XGBOOST e MultiLayerPerceptron**
#
`
Esse projeto tem como foco usar uma base de dados totalmente desbalanceada para verificar o comportamento dos algorítmos 
`

A base de dados utilizada encontra-se no meu google-drive, entretanto,
o mesmo está sendo disponibilizado no repositório do kaggle: "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud"

In [ ]:
# Montando o drive onde se encontra o data set;
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Criando o diretório onde será colocado o dataset
!mkdir dataset
!cp drive/MyDrive/creditcard.csv dataset

In [ ]:
import tensorflow as tf
print("GPUs disponíveis:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Importações necessárias para o projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, regularizers
from keras.metrics import Precision, Recall, AUC
from keras.optimizers import Adam

In [ ]:
# Leitura inicial dos dados do dataset
df = pd.read_csv("dataset/creditcard.csv")

In [ ]:
# Verificando os atributos do dataset
print(len(df.columns))
print("Quantidade de dados:",len(df))


In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
print(df.select_dtypes(include=object))

`
OBS: Como pode ser notado, não existem dados categóricos nesse dataset
`

In [ ]:
# Criação da matriz de correlação de features em escolhidas no dataset
conjunto_cols = df.columns.to_series().sample(10).tolist()

if "Class" not in conjunto_cols:
    conjunto_cols.append("Class")

# Correlações entre as variáveis numéricas
plt.figure(figsize=(25, 12))
sns.heatmap(df[conjunto_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlação entre variáveis numéricas")
plt.show()

In [ ]:
# Copiando o dataset para uma nova variável new_df
new_df = df.copy()
new_df.isna().sum() >=1 # Validadendo se há valores NaN ou nulos de um modo geral

In [ ]:
# Fazendo a contagem de quantos dos dados são positivos ou negativos
fraud_cases = new_df[new_df["Class"] == 1]
not_fraud_cases = new_df[new_df["Class"] == 0]

print(f"Total de casos de fraude: {len(fraud_cases)}")
print(f"Total de casos não fraude: {len(not_fraud_cases)}")

# Separando dados de treino e teste

In [ ]:

# Separando as features e o target
X = new_df.drop("Class", axis=1)
y = new_df["Class"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
# Separando um conjunto a parte para validação - o conjunto é retirado do treinamento
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42
)


`
Como fora observado, os dados estão desbalanceados. Dessa forma, essa etapa pega os pesos de cada uma das classes
`

In [ ]:
# A função create_model é responsável pela criação do modelo
def create_model(dropout_rate=0.5, l2_reg=1e-4):
    """Parâmetros:
       dropout_rate: taxa de dropout que as camadas devem fazer
       l2_reg: Parâmetro de regularização - valor do parâmetro lambda
    """

    model = keras.Sequential([
        layers.Dense(
            1024,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.95),

        layers.Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.9),

        layers.Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.8),

        layers.Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.7),

        layers.Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.6),

        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.5),

        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate * 0.4),

        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),

        layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg)
        ),

        layers.Dense(1, activation="sigmoid")
    ])

    return model

In [ ]:
# model - o modelo que será usado
model = create_model(
    dropout_rate=0.3,
    l2_reg=0.0001
)

# Compilando o modelo com o otimizador Adam, e a perda binária de entropia cruzada
model.compile(
    optimizer=Adam(learning_rate=0.00009),
    loss="binary_crossentropy",
    metrics=["accuracy", Precision(), Recall(), AUC()]
)

In [ ]:
# Fazendo a separação das colunas que precisarão ser normalizadas
normalize_cols = X_train.columns.to_list()
normalize_cols

In [ ]:
# pipeline de normalização
num_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Transformação das colunas a serem normalizadas
preprocessor = ColumnTransformer(
    transformers=[
        ("num_norm", num_pipeline, normalize_cols)
    ],
    remainder="passthrough"
)


In [ ]:
# Pipeline final de treinamento
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model) # Algorítmo de redes neurais
])


In [ ]:
preprocessor = pipeline.named_steps["preprocessor"]

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)


# Treinamento do modelo
**OBSERVAÇÃO:**
`
Algo importante é que, apesar da quantidade de épocas usada, o modelo está usando o EarlyStopping.
Além disso, esse é o modelo final - Vários testes foram feitos na MLP e aqui apresentou o "melhor" resultado, foi essa para competir com XGBOOST
`

In [ ]:
# Treinando o modelo
model = pipeline.named_steps["model"]

model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=15,
    callbacks=[EarlyStopping(
        patience=5,
        restore_best_weights=True
    )]
)



**Algo que eu considero de alta relevância é a plotagem da curva de perda da validação e treino ao longo das épocas**
`
Assim, posso verificar se o modelo está com variância e válidar o decaimento, se foi
normal ou com muitas oscilações
`

In [ ]:
history = model.history.history

# Vizualização da perda do modelo ao longo das épocas
plt.figure(figsize=(12, 5))
plt.plot(history["loss"], label="Perda de Treino")
plt.plot(history["val_loss"], label="Perda de Validação")
plt.title("Perda do Modelo ao Longo das Épocas")
plt.xlabel("Épocas")
plt.ylabel("Perda")
plt.legend()
plt.show()

In [ ]:
# transformando o dicionário de
hist = list(history.keys())
hist

**Curva de recall - treino e validação**
`
Esse caso é idêntico ao anterior,
a visualização da curva de recall no treino e validação
é uma ótima ferramenta para validar se há variância no modelo.
`

In [ ]:
# Visualização do recall do modelo ao longo das épocas
plt.figure(figsize=(12, 5))
plt.plot(history[hist[4]], label="Recall de Treino")
plt.plot(history[hist[9]], label="Recall de Validação")
plt.title("Recall do Modelo ao Longo das Épocas")
plt.xlabel("Épocas")
plt.ylabel("Recall")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score, precision_score

best_t = 0
best_f1 = 0

y_val_prob = model.predict(X_val).ravel()

plt.hist(y_val_prob[y_val==0], bins=50, alpha=0.5)
plt.hist(y_val_prob[y_val==1], bins=50, alpha=0.5)

plt.legend()
plt.title("Distribuição das Probabilidades")
plt.show()



In [ ]:
# Previsão do modelo em relação ao teste
y_pred = model.predict(X_test)
y_pred_classes = (y_pred >= 0.1).astype(int) # Ajustando o limiar para 0.3 para tentar normalizar o precision e recall

# Avaliação das métricas
**Agora, será feito a avaliação do desempenho do modelo em seu conjunto de testes**

In [ ]:
# Vizualizando a matriz de confusão
plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_classes), annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusão")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

In [ ]:
# Instanciando a Precision, recall e AUC
precision = Precision()
recall = Recall()
auc = AUC()

In [ ]:
# Pegando os valores das métricas para avaliação
precision.update_state(y_test, y_pred_classes)
recall.update_state(y_test, y_pred_classes)
auc.update_state(y_test, y_pred_classes)

In [ ]:
# Vizualização das métricas
print(f"Precision: {precision.result().numpy():.4f}")
print(f"Recall: {recall.result().numpy():.4f}")
print(f"AUC: {auc.result().numpy():.4f}")

In [ ]:
y_prob = model.predict(X_test).ravel()

In [ ]:
# Vizualizando a curva roc - o quanto o modelo detecta fraudes (TPR) e o quanto acusa fraude errada (FPR)
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
# A curva roc pode enganar um pouco - apesar de seu grande uso. Então, usei o Precision-Recall Curve
"Vizualizando a curva da precisão e do recall"

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

# XGBOOST
`Nessa etapa, vamos testar os poderes do algorítmo XGBOOST`

In [ ]:
# O modelo nesse momento não terá o earlystopping
# A decisão foi tomada por que será usada o RandomizedSearchCV - que irá encontrar os melhores parâmetros para o modelo
# Mas, na etapa de refinamento (que será apenas uma, para não ser injusto com o algorítmo de MLP) será usado com a mesma patience do MLP
modelo_xg = XGBClassifier(
    tree_method="hist",
)

In [ ]:
# Pipeline para normalização - igual o anterior - usando StandardScaler
norm_pipeline_xg = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Preprocessador das colunas que precisam ser normalizadas
preprocessor_xg = ColumnTransformer(
    transformers=[
        ("num_norm", num_pipeline, normalize_cols)
    ],
    remainder="passthrough"
)


In [ ]:
pipe_xg = Pipeline(steps=[
    ("preprocessor", preprocessor_xg),
    ("model", modelo_xg)
])


In [ ]:
param_dist = {
    'model__n_estimators': [900, 1000],
    'model__max_depth': [4, 5, 6],
    'model__learning_rate': [0.006, 0.01],
    'model__subsample': [0.5, 0.6],
    'model__colsample_bytree': [0.7, 0.8]
}


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
# Pesquisa aleatória dos melhores parâmetros para o nosso xgboost
random_search = RandomizedSearchCV(
    estimator=pipe_xg,
    param_distributions=param_dist,
    n_iter=15, # as iterações serão com base na quantidade de iterações feitas pela MLP
    scoring='roc_auc',
    cv=skf,
    verbose=2,
    random_state=42,
    n_jobs=-1
)


# Treinamento do modelo

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
# Capturando o melhor modelo escolhido pelo random_search
best_model = random_search.best_estimator_
y_pred_xg = best_model.predict(X_test)

In [ ]:
# Como foi usado 0.1 no MLP, aqui também será feito a mesma coisa
y_pred_classes_xg = (y_pred_xg >= 0.1).astype(int)

# Visualização das Métricas

In [ ]:
# Vizualização da matriz de confusão
plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_classes_xg), annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusão")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()


In [ ]:
# Instanciando as métricas usadas
precision_xg = Precision()
recall_xg = Recall()
auc_xg = AUC()

In [ ]:
# Pegando as métricas
precision_xg.update_state(y_test, y_pred_classes_xg)
recall_xg.update_state(y_test, y_pred_classes_xg)
auc_xg.update_state(y_test, y_pred_classes_xg)

In [ ]:
# Vizualizando as métricas
print(f"Precision: {precision_xg.result().numpy():.4f}")
print(f"Recall: {recall_xg.result().numpy():.4f}")
print(f"AUC: {auc_xg.result().numpy():.4f}")

In [ ]:
# Pegando as probabilidades do modelo
y_prob_xg = y_pred_xg.ravel()

In [ ]:
# Vizualizando a curva roc para o xgboost
fpr, tpr, _ = roc_curve(y_test, y_prob_xg)
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
#"Vizualizando a curva da precisão e do recall"
precision, recall, _ = precision_recall_curve(y_test, y_prob_xg)
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

# Análise de performance dos algorítmos usados
Problemas de classificação envolvendo dados altamente desbalanceados são extremamente comuns em aplicações do mundo real, como detecção de fraudes financeiras, diagnósticos médicos e detecção de anomalias em sistemas críticos.

Nestes cenários, a classe de interesse (por exemplo, fraudes) geralmente representa uma fração muito pequena do conjunto total de dados — o que dificulta significativamente o processo de aprendizado por parte dos modelos de Machine Learning.

Modelos tradicionais, como redes neurais do tipo Multilayer Perceptron (MLP), tendem a apresentar dificuldades em aprender padrões relevantes da classe minoritária quando treinados diretamente sobre dados desbalanceados, muitas vezes resultando em classificações enviesadas em direção à classe majoritária.

Diante disso, este trabalho teve como objetivo comparar o desempenho de uma rede neural MLP com o algoritmo XGBoost, avaliando sua capacidade de generalização em um cenário de detecção de fraudes sem a aplicação de técnicas de balanceamento de dados.




📊 Resultados Obtidos

Após o treinamento dos modelos diretamente sobre o conjunto de dados original (contendo aproximadamente 0.116% de amostras positivas), foram obtidas as seguintes matrizes de confusão:

**MLP**: [22, 56842]
     [0,    98]

`
Apesar de apresentar recall máximo (1.0), indicando que todas as fraudes foram corretamente identificadas, o modelo classificou erroneamente 56.842 transações legítimas como fraudulentas.
Na prática, isso implicaria no bloqueio de milhares de transações válidas para cada fraude real detectada, tornando o modelo inviável para aplicações reais.
`

**XGBOOST**: [56859, 5]
         [20,   78]

Mesmo sem qualquer técnica de balanceamento aplicada, o modelo baseado em árvores de decisão demonstrou:
  * Redução drástica de falsos positivos (apenas 5)
  * Elevada precisão (~94%)
  * Capacidade de identificar aproximadamente 80% das fraudes

 Isso indica que o algoritmo foi capaz de aprender padrões discriminativos relevantes da classe minoritária sem comprometer significativamente o desempenho sobre a classe majoritária.

# ANÁLISE PESSOAL:
*Embora o MLP tenha alcançado um recall superior, seu número excessivo de falsos positivos evidencia um comportamento de colapso em direção à classe minoritária — um problema comum em redes neurais treinadas sobre dados desbalanceados.
Por outro lado, o XGBoost demonstrou maior robustez, mantendo um equilíbrio entre sensibilidade e especificidade, além de apresentar maior capacidade de generalização mesmo diante da distribuição assimétrica dos dados.
Esses resultados sugerem que algoritmos baseados em árvores de decisão, especialmente aqueles que utilizam técnicas de Gradient Boosting, podem ser mais adequados para cenários onde o desbalanceamento é significativo e a detecção da classe minoritária é crítica.*

✅ Conclusão

A partir dos experimentos realizados, foi possível observar que o XGBoost apresentou desempenho superior ao MLP no contexto de dados altamente desbalanceados, mesmo sem a utilização de métodos adicionais de balanceamento.

Dessa forma, pode-se considerar o XGBoost como uma alternativa mais robusta e eficaz para tarefas de classificação em cenários reais de detecção de fraudes.

🙏 Agradecimentos

Agradeço ao Professor Dr. Eanes Torres Pereira pela motivação contínua no estudo de Machine Learning e pela pasciência em responder minhas dúvidas haha.